# BSE API
We are going to develop a small tool to get basis sets from the Basis Set Exchange. 

In [1]:
import requests
from typing import List, Tuple

In [2]:
def get_basis_set(element_number_list: List[int], basis_set:str) -> str:
    """ 
    Fetches the specified basis set for given elements from the Basis Set Exchange (BSE) API.

    Parameters
    ----------
    element_number_list : List[int]
        A list of atomic numbers representing the elements for which the basis set is requested.
    basis_set : str
        The name of the basis set to be fetched (e.g., 'sto-3g', '6-31g', etc.).
    
    Returns
    -------
    str
        The basis set data in NWChem format as a string.
    """

    try:
        element_number_list = [str(i) for i in element_number_list]
        elements = ",".join(element_number_list)


        url = f"https://www.basissetexchange.org/api/basis/{basis_set}/format/nwchem/?version=1&elements={elements}"

        response = requests.get(url, timeout=10)


        response.raise_for_status()

        content = response.text

        return content
    
    except requests.exceptions.RequestException as e:
        print(f"Combination of basis {basis_set} and elements {elements} was not found in the Basis Set Exchange.")

In [3]:
element_number_list = [27,1]
basis_set = '6-31g'

basis_1 = get_basis_set(element_number_list, basis_set)

In [4]:
def locate_indices(basis_list: list[str]) -> Tuple[int]:
    list_of_indices = []
    for i, line in enumerate(basis_list):
        if '#BASIS SET:' in line:
            list_of_indices.append(i)
    
    list_of_indices.append(len(basis_list)-2)

    return list_of_indices

In [5]:
basis_list = basis_1.split('\n')
indices = locate_indices(basis_list)
indices

[13, 20, 54]

In [6]:
print("\n".join(basis_list[indices[0]:indices[1]]))

#BASIS SET: (4s) -> [2s]
H    S
      0.1873113696E+02       0.3349460434E-01
      0.2825394365E+01       0.2347269535E+00
      0.6401216923E+00       0.8137573261E+00
H    S
      0.1612777588E+00       1.0000000


In [7]:
print("\n".join(basis_list[indices[1]:indices[2]]))

#BASIS SET: (22s,16p,4d) -> [5s,4p,2d]
Co    S
      0.6614899000E+05       0.1759787106E-02
      0.9933077000E+04       0.1348162081E-01
      0.2262816000E+04       0.6649342399E-01
      0.6379154000E+03       0.2307939139E+00
      0.2044122000E+03       0.4792919288E+00
      0.6982538000E+02       0.3514097211E+00
Co    SP
      0.1378841000E+04       0.2376276103E-02       0.3971488140E-02
      0.3282694000E+03       0.3167450137E-01       0.3108174109E-01
      0.1060946000E+03       0.1262888054E+00       0.1357439048E+00
      0.3983275000E+02      -0.2584552112E-01       0.3476827122E+00
      0.1618622000E+02      -0.6183491267E+00       0.4626340163E+00
      0.6667788000E+01      -0.4567008197E+00       0.2051632072E+00
Co    SP
      0.5452355000E+02      -0.3993003860E-02      -0.7290771623E-02
      0.1829783000E+02       0.7409662740E-01      -0.2926026849E-01
      0.7867348000E+01       0.2541999911E+00       0.6564149661E-01
      0.3340534000E+01      -0.2921656

In [8]:
def extract_element_basis(basis_list, indices) -> dict:
    element_str = basis_list[indices[0]+1].strip().split()[0]

    shell = 0
    shell_dict = {shell:[]}
    for line in basis_list[indices[0] + 2 : indices[1]]:


        try:
            float(line.strip().split()[0])
            shell_dict[shell].append([float(i) for i in line.strip().split()])

        except:
            shell += 1
            shell_dict.update({shell:[]})

    return {element_str: shell_dict}

In [9]:
total_basis = {}
for i in range(len(indices)-1):
    total_basis.update(extract_element_basis(basis_list, (indices[i], indices[i+1])))

total_basis
 

{'H': {0: [[18.73113696, 0.03349460434],
   [2.825394365, 0.2347269535],
   [0.6401216923, 0.8137573261]],
  1: [[0.1612777588, 1.0]]},
 'Co': {0: [[66148.99, 0.001759787106],
   [9933.077, 0.01348162081],
   [2262.816, 0.06649342399],
   [637.9154, 0.2307939139],
   [204.4122, 0.4792919288],
   [69.82538, 0.3514097211]],
  1: [[1378.841, 0.002376276103, 0.00397148814],
   [328.2694, 0.03167450137, 0.03108174109],
   [106.0946, 0.1262888054, 0.1357439048],
   [39.83275, -0.02584552112, 0.3476827122],
   [16.18622, -0.6183491267, 0.4626340163],
   [6.667788, -0.4567008197, 0.2051632072]],
  2: [[54.52355, -0.00399300386, -0.007290771623],
   [18.29783, 0.0740966274, -0.02926026849],
   [7.867348, 0.2541999911, 0.06564149661],
   [3.340534, -0.2921656898, 0.4000651793],
   [1.393756, -0.7318702744, 0.4950235744],
   [0.551326, -0.2040783929, 0.1758239909]],
  3: [[2.151947, 0.05379840456, -0.2165495935],
   [0.811063, 0.2759969695, 0.1240487963],
   [0.121017, -1.129691466, 0.9724063708]

In [10]:
def index_elements(element_list):
    element_list = set(element_list)
    element_indices = [element_dict(i) for i in element_list]

    return element_indices


def element_dict(element_str):
    elem_dict = {
        "H": 1,
        "He": 2,
        "Li": 3,
        "Be": 4,
        "B": 5,
        "C": 6,
        "N": 7,
        "O": 8,
        "F": 9,
        "Ne": 10,
        "Na": 11,
        "Mg": 12,
        "Al": 13,
        "Si": 14,
        "P": 15,
        "S": 16,
        "Cl": 17,
        "Ar": 18,
        "K": 19,
        "Ca": 20,
        "Sc": 21,
        "Ti": 22,
        "V": 23,
        "Cr": 24,
        "Mn": 25,
        "Fe": 26,
        "Co": 27,
        "Ni": 28,
        "Cu": 29,
        "Zn": 30,
        "Ga": 31,
        "Ge": 32,
        "As": 33,
        "Se": 34,
        "Br": 35,
        "Kr": 36,
        "Rb": 37,
        "Sr": 38,
        "Y": 39,
        "Zr": 40,
        "Nb": 41,
        "Mo": 42,
        "Tc": 43,
        "Ru": 44,
        "Rh": 45,
        "Pd": 46,
        "Ag": 47,
        "Cd": 48,
        "In": 49,
        "Sn": 50,
        "Sb": 51,
        "Te": 52,
        "I": 53,
        "Xe": 54,
        "Cs": 55,
        "Ba": 56,
        "La": 57,
        "Ce": 58,
        "Pr": 59,
        "Nd": 60,
        "Pm": 61,
        "Sm": 62,
        "Eu": 63,
        "Gd": 64,
        "Tb": 65,
        "Dy": 66,
        "Ho": 67,
        "Er": 68,
        "Tm": 69,
        "Yb": 70,
        "Lu": 71,
        "Hf": 72,
        "Ta": 73,
        "W": 74,
        "Re": 75,
        "Os": 76,
        "Ir": 77,
        "Pt": 78,
        "Au": 79,
        "Hg": 80,
        "Tl": 81,
        "Pb": 82,
        "Bi": 83,
        "Po": 84,
        "At": 85,
        "Rn": 86,
        "Fr": 87,
        "Ra": 88,
        "Ac": 89,
        "Th": 90,
        "Pa": 91,
        "U": 92,
        "Np": 93,
        "Pu": 94,
        "Am": 95,
        "Cm": 96,
        "Bk": 97,
        "Cf": 98,
        "Es": 99,
        "Fm": 100,
        "Md": 101,
        "No": 102,
        "Lr": 103,
        "Rf": 104,
        "Db": 105,
        "Sg": 106,
        "Bh": 107,
        "Hs": 108,
        "Mt": 109,
        "Ds": 110,
        "Rg": 111,
        "Cn": 112,
        "Nh": 113,
        "Fl": 114,
        "Mc": 115,
        "Lv": 116,
        "Ts": 117,
        "Og": 118,
    }
    return elem_dict[element_str]


def fetch_parsed_basis(element_list, basis_name):
    elements_to_fetch = index_elements(element_list)
    basis_str = get_basis_set(elements_to_fetch, basis_name)
    basis_list = basis_str.split("\n")
    indices = locate_indices(basis_list)
    total_basis = {}
    for i in range(len(indices) - 1):
        total_basis.update(
            extract_element_basis(basis_list, (indices[i], indices[i + 1]))
        )

    return total_basis

In [11]:
fetch_parsed_basis(['Li', 'He'], '6-31g')

{'He': {0: [[38.421634, 0.04013973935],
   [5.77803, 0.261246097],
   [1.241774, 0.7931846246]],
  1: [[0.297964, 1.0]]},
 'Li': {0: [[642.418915, 0.00214260781],
   [96.7985153, 0.0162088715],
   [22.0911212, 0.0773155725],
   [6.20107025, 0.245786052],
   [1.93511768, 0.470189004],
   [0.636735789, 0.345470845]],
  1: [[2.324918408, -0.03509174574, 0.008941508043],
   [0.6324303556, -0.1912328431, 0.141009464],
   [0.07905343475, 1.083987795, 0.9453636953]],
  2: [[0.03596197175, 1.0, 1.0]]}}